# Linear Algebra with SciRS2

This notebook demonstrates the linear algebra capabilities of `scirs2-linalg`,
which mirrors `scipy.linalg` and `numpy.linalg`. All operations are implemented
in pure Rust using OxiBLAS (no system BLAS/LAPACK required).

Topics covered:
- Solving linear systems (Ax = b)
- LU, QR, and Cholesky decompositions
- Eigenvalue decomposition
- Singular Value Decomposition (SVD)
- Matrix functions (exponential, logarithm)
- Iterative solvers for large systems

In [ ]:
:dep scirs2-core = { path = "../scirs2-core" }
:dep scirs2-linalg = { path = "../scirs2-linalg" }

In [ ]:
use scirs2_core::ndarray::{array, Array2};
use scirs2_linalg::{det, inv, solve, lu, qr, svd, cholesky, eig};

## Solving Linear Systems

Given a system Ax = b, we can solve for x directly using `solve`,
which internally performs LU decomposition with partial pivoting.

In [ ]:
// System of 3 equations in 3 unknowns
//   2x + y - z  = 8
//  -3x - y + 2z = -11
//  -2x + y + 2z = -3
let a = array![[ 2.0,  1.0, -1.0],
               [-3.0, -1.0,  2.0],
               [-2.0,  1.0,  2.0]];
let b = array![8.0, -11.0, -3.0];

let x = solve(&a.view(), &b.view(), None).expect("solve failed");
println!("Solution: x = {}", x);
println!("Expected: [2, 3, -1]");

// Verification
let residual = &a.dot(&x) - &b;
println!("Residual norm: {:.2e}", residual.mapv(|v| v * v).sum().sqrt());

## LU Decomposition

LU decomposition factors A = PLU where P is a permutation matrix,
L is lower triangular with unit diagonal, and U is upper triangular.
This is the workhorse behind `solve`.

In [ ]:
let a = array![[2.0, 1.0],
               [5.0, 3.0]];

let (p, l, u) = lu(&a.view(), None).expect("LU failed");
println!("P =\n{}", p);
println!("L =\n{}", l);
println!("U =\n{}", u);

// Verify: P * A = L * U
let pa = p.dot(&a);
let lu_product = l.dot(&u);
println!("\nP*A =\n{}", pa);
println!("L*U =\n{}", lu_product);

## QR Decomposition

QR decomposition factors A = QR where Q is orthogonal and R is upper
triangular. Useful for least-squares problems and eigenvalue algorithms.

In [ ]:
let a = array![[1.0, 2.0],
               [3.0, 4.0],
               [5.0, 6.0]];

let (q, r) = qr(&a.view(), None).expect("QR failed");
println!("Q =\n{:.4}", q);
println!("R =\n{:.4}", r);

// Verify orthogonality: Q^T * Q should be identity
let qtq = q.t().dot(&q);
println!("\nQ^T * Q (should be ~I) =\n{:.6}", qtq);

## Eigenvalue Decomposition

For a square matrix A, eigenvalue decomposition finds scalars lambda
and vectors v such that Av = lambda * v.

In [ ]:
// Symmetric matrix (guaranteed real eigenvalues)
let a = array![[4.0, 1.0],
               [1.0, 3.0]];

let (eigenvalues, eigenvectors) = eig(&a.view(), None).expect("eig failed");
println!("Eigenvalues: {}", eigenvalues);
println!("Eigenvectors:\n{}", eigenvectors);

// For a symmetric 2x2 [[4,1],[1,3]], eigenvalues are (7+sqrt(5))/2 and (7-sqrt(5))/2
let expected_1 = (7.0 + 5.0_f64.sqrt()) / 2.0;
let expected_2 = (7.0 - 5.0_f64.sqrt()) / 2.0;
println!("\nExpected eigenvalues: {:.6}, {:.6}", expected_1, expected_2);

## Singular Value Decomposition (SVD)

SVD factors any matrix A = U * diag(S) * V^T. It is fundamental for
dimensionality reduction (PCA), matrix approximation, and pseudoinverses.

In [ ]:
let a = array![[1.0, 0.0, 0.0],
               [0.0, 2.0, 0.0],
               [0.0, 0.0, 3.0]];

let (u, s, vt) = svd(&a.view(), true, None).expect("SVD failed");
let u = u.expect("U matrix");
let vt = vt.expect("Vt matrix");

println!("Singular values: {}", s);
println!("U =\n{:.4}", u);
println!("V^T =\n{:.4}", vt);

In [ ]:
// Low-rank approximation: keep only the top-k singular values
let a = array![[1.0, 2.0, 3.0],
               [4.0, 5.0, 6.0],
               [7.0, 8.0, 9.0]];

let (u_opt, s, vt_opt) = svd(&a.view(), true, None).expect("SVD failed");
let u = u_opt.expect("U");
let vt = vt_opt.expect("Vt");

println!("Singular values: {}", s);
println!("Note: the third singular value is ~0 (rank-2 matrix)");

// Rank-1 approximation: use only the largest singular value
let u_col0 = u.column(0).to_owned();
let vt_row0 = vt.row(0).to_owned();
let rank1 = s[0] * u_col0.into_shape((3, 1)).expect("reshape")
    .dot(&vt_row0.into_shape((1, 3)).expect("reshape"));
println!("\nRank-1 approximation:\n{:.4}", rank1);

## Cholesky Decomposition

For symmetric positive definite matrices, Cholesky factorization gives
A = L * L^T where L is lower triangular. It is roughly twice as efficient
as LU and is the preferred method for SPD systems.

In [ ]:
// Symmetric positive definite matrix
let spd = array![[4.0, 2.0, 1.0],
                 [2.0, 5.0, 3.0],
                 [1.0, 3.0, 6.0]];

let l = cholesky(&spd.view(), None).expect("Cholesky failed");
println!("L =\n{:.4}", l);

// Verify: L * L^T = A
let llt = l.dot(&l.t());
println!("\nL * L^T =\n{:.4}", llt);
println!("Original =\n{:.4}", spd);

## Iterative Solvers

For large sparse systems, iterative methods are more efficient than direct
factorization. Conjugate Gradient works for SPD systems, while GMRES handles
general non-symmetric systems.

In [ ]:
use scirs2_linalg::{conjugate_gradient, gmres};

// SPD system for Conjugate Gradient
let a = array![[10.0,  1.0,  0.0],
               [ 1.0, 10.0,  1.0],
               [ 0.0,  1.0, 10.0]];
let b = array![11.0, 12.0, 11.0];

let x_cg = conjugate_gradient(&a.view(), &b.view(), 100, 1e-12, None)
    .expect("CG failed");
println!("CG solution: {}", x_cg);

// GMRES for general systems
let x_gmres = gmres(&a.view(), &b.view(), 100, 1e-12, None)
    .expect("GMRES failed");
println!("GMRES solution: {}", x_gmres);

// Both should give the same answer
let residual = (&x_cg - &x_gmres).mapv(|v| v.abs()).sum();
println!("Difference between CG and GMRES: {:.2e}", residual);

## Notes on Performance

SciRS2 linear algebra is built on **OxiBLAS**, a pure-Rust BLAS implementation.
Key points:

- No C/Fortran dependencies -- the entire stack compiles with `cargo build`
- SIMD acceleration (AVX2/AVX-512/NEON) is used where available
- For very large matrices, enable the `parallel` feature for multi-threaded operation
- Performance is competitive with OpenBLAS for typical scientific workloads

For production use, consider enabling features in `Cargo.toml`:
```toml
scirs2-linalg = { version = "0.4.0", features = ["simd", "parallel"] }
```